# INT8 Quantized Linear

**难度:** Hard

实现 INT8 逐通道权重量化线性层。

量化将 float32 权重转换为 int8 并使用逐通道缩放，将模型大小减少 4 倍同时保持精度。

**签名:** `Int8Linear(weight, bias=None)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (*, in_features)

**返回:** 使用反量化权重的线性输出

**约束:**
- 逐通道缩放：`abs(weight).max(dim=1) / 127`
- 量化：`round(weight/scale).clamp(-128, 127).to(int8)`
- weight_int8 和 scale 存储为 buffer 而非 parameter

**提示:**
1. scale = abs(weight).max(dim=1) / 127（逐输出通道，keepdim）
2. weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)
3. 两者均注册为 buffer（非 parameter）
4. 前向：dequant = weight_int8.float() * scale → x @ dequant.T

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        super().__init__()

        # ---- Step 1: 计算逐通道缩放因子 ----
        # 每个输出通道（weight 的每一行）独立计算 scale
        # scale = max(|w|) / 127，确保量化后最大值刚好填满 int8 范围
        # keepdim=True 保持 (out_features, 1) 的 shape，方便后续广播除法
        scale = weight.abs().amax(dim=1, keepdim=True) / 127.0

        # ---- Step 2: 量化为 int8 ----
        # round 将浮点值四舍五入为整数
        # clamp(-128, 127) 确保在 int8 范围内
        # register_buffer 注册为 buffer：随模型保存/加载，但不参与梯度计算
        self.register_buffer('weight_int8',
            torch.round(weight / (scale + 1e-10)).clamp(-128, 127).to(torch.int8))
        self.register_buffer('scale', scale)

        # bias 保持 float32，注册为 parameter（如果有的话）
        self.bias = nn.Parameter(bias.clone()) if bias is not None else None

    def forward(self, x):
        # ---- Step 3: 反量化 + 线性计算 ----
        # 反量化：int8 * scale 恢复为 float32（有损，但误差很小）
        w = self.weight_int8.float() * self.scale
        # 矩阵乘法
        out = x @ w.T
        if self.bias is not None:
            out = out + self.bias
        return out

In [ ]:
# Demo
w = torch.randn(8, 4)
q = Int8Linear(w)
print('Output:', q(torch.randn(2, 4)).shape)
print('Weight dtype:', q.weight_int8.dtype)
print('Compression: float32 -> int8 = 4x')

In [ ]:
from torch_judge import check
check('int8_quantization')

## 📝 核心思路总结

1. **量化 = 浮点→整数映射**：用 scale 因子将 float32 映射到 int8 范围 [-128, 127]
2. **逐通道缩放保持精度**：每个输出通道独立计算 scale，避免少量大值"压缩"小值
3. **buffer vs parameter**：量化后的权重不参与训练，注册为 buffer 而非 parameter